In [13]:
from IPython.display import display, Math
import sympy as sp
from fractions import Fraction
from typing import List, Tuple, Union
import numpy as np
import pandas as pd

pd.set_option('display.precision', 12)  # Increase decimal precision
pd.set_option('display.width', 300)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column


In [14]:
def read_matrix(path: str) -> pd.DataFrame:
    data = []
    with open(path, 'r') as f:
        for lineno, line in enumerate(f, start=1):
            tokens = line.strip().split()
            if not tokens:
                continue
            row = []
            for tok in tokens:
                try:
                    if tok.count('/') == 1:
                        num, den = tok.split('/')
                        val = int(num) / int(den)
                    else:
                        val = float(tok)
                except Exception as e:
                    raise ValueError(f"Error parsing token '{tok}' on line {lineno} in {path}: {e}")
                row.append(val)
            data.append(row)
    if not data:
        return pd.DataFrame()
    ncol = len(data[0])
    if any(len(r) != ncol for r in data):
        lengths = [len(r) for r in data]
        raise ValueError(f"Row-length mismatch in {path}: expected {ncol} columns, got {lengths}")
    return pd.DataFrame(data)
def output_matrix(mat, precision=4, display_obj=True):
    if isinstance(mat, pd.DataFrame):
        mat = mat.values
    if not isinstance(mat, np.ndarray):
        mat = np.array(mat)
    def format_val(val):
        if np.isrealobj(val) or np.isreal(val):
            return f"{np.real(val):.{precision}f}"
        else:
            r = np.real(val)
            i = np.imag(val)
            if abs(i) < 1e-10:
                return f"{r:.{precision}f}"
            sign = "+" if i >= 0 else "-"
            return f"{r:.{precision}f} {sign} {abs(i):.{precision}f}i"
    formatted_mat = np.vectorize(format_val)(mat)
    sp_mat = sp.Matrix(formatted_mat)
    latex_str = sp.latex(sp_mat)
    
    if display_obj:
        display(Math(latex_str))
    return latex_str


In [15]:
# input
A_df = read_matrix("SVD_input_A.txt")


# Lũy thừa + Xuống thang

In [16]:
def deflate_once(A: np.ndarray, lam: float, v: np.ndarray, x0_left: np.ndarray, tol: float, max_iter: int, norm_ord: Union[int, float], precision: int) -> np.ndarray:
    _, w = power_method(
        A.T, x0_left, tol=tol, max_iter=max_iter,
        norm_ord=norm_ord, precision=precision, display=False
    )
    scale = w.dot(v)
    if abs(scale) < 1e-12:
        raise ValueError("Left and right eigenvectors nearly orthogonal; cannot deflate.")
    w = w / scale
    return A - lam * np.outer(v, w)


In [17]:
def power_method(A,
                 x0,
                 tol=1e-6,
                 max_iter=1000,
                 norm_ord=2,
                 precision=12,
                 display=True):
    """
    Power Method for finding the dominant eigenvalue and eigenvector of A.
    Displays each iteration in a pandas DataFrame using the specified norm.

    Parameters:
      A (ndarray): Square matrix (n×n).
      x0 (ndarray): Initial guess vector (n,).
      tol (float): Convergence tolerance for eigenvalue change.
      max_iter (int): Maximum number of iterations.
      norm_ord (int, float, or str): Norm order for both eigenvalue estimation and vector normalization (e.g. 1, 2, np.inf).
      precision (int): Number of decimal places to display.

    Returns:
      eigenvalue (float): Approximate dominant eigenvalue.
      eigenvector (ndarray): Approximate eigenvector corresponding to the dominant eigenvalue.
    """
    # Normalize initial guess
    x = x0 / np.linalg.norm(x0, ord=norm_ord)

    # Step 0: record the very first y,x,λ
    y0 = x0           # or if you want y0=x0 then skip this dot
    lambda0 = np.linalg.norm(y0, ord=norm_ord)
    x0_norm = y0 / lambda0

    # history entries: [iteration, y1…yn, lambda, x1…xn]
    history = [
      [0] + y0.tolist() + [lambda0] + x0_norm.tolist()
    ]

    eigenvalue = lambda0
    n = A.shape[0]
    

    for k in range(1, max_iter+1):
        # Compute A x_{k-1}
        y = A.dot(x)
        # Estimate eigenvalue via the chosen norm
        lambda_new = np.linalg.norm(y, ord=norm_ord)
        if lambda_new == 0:
            raise ZeroDivisionError("Norm of A*x is zero; unable to continue.")
        # Normalize to get next eigenvector estimate
        x_new = y / lambda_new

        # Record iteration data: [iter, y1..yn, lambda, x1..xn]
        row = [k] + y.tolist() + [lambda_new] + x_new.tolist()
        history.append(row)

        # Check convergence on eigenvalue
        if abs(lambda_new - eigenvalue) < tol:
            break

        x = x_new
        eigenvalue = lambda_new

    # Build and display DataFrame
    cols = ['Iteration'] + [f'y{i+1}' for i in range(n)] + ['lambda'] + [f'x{i+1}' for i in range(n)]
    df = pd.DataFrame(history, columns=cols)
    if (display==True):
      print(df.to_string(
        index=False,
        float_format=lambda v: f"{v:.{precision}f}"
    ))

    return eigenvalue, x_new


In [18]:
def power_method_case2(
    A,
    x0,
    tol=1e-6,
    max_iter=1000,
    norm_ord=np.inf,
    precision=12,
    display=True
):
    """
    Power Method for when the two largest eigenvalues satisfy |λ1| = |λ2| > |λ3| and λ1 = -λ2.
    Uses even powers of A to separate eigencomponents, printing y and x in one table.

    Parameters:
      A (ndarray): Square matrix (n×n).
      x0 (ndarray): Initial guess vector (n,).
      tol (float): Convergence tolerance for λ^2 change.
      max_iter (int): Maximum number of iterations for k.
      norm_ord (int, float, or str): Norm order for vector normalization.
      precision (int): Decimal places for display.

    Returns:
      lambda1 (float): Approximate dominant eigenvalue (positive).
      x1 (ndarray): Eigenvector corresponding to λ1.
      x2 (ndarray): Eigenvector corresponding to λ2 = -λ1.
    """
    n = A.shape[0]

    # Step 0: input vector and its norm^2
    y0 = x0.astype(float)
    lambda0 = np.linalg.norm(y0, ord=norm_ord)
    lambda2_old = lambda0**2
    x0_norm = y0 / lambda0

    history = []
    # record iteration 0
    history.append(
      [0] + y0.tolist() + [lambda2_old] + x0_norm.tolist() + x0_norm.tolist()
    )

    # now do the usual even‐odd power iterations
    y_even = x0.astype(float)


    for k in range(1, max_iter + 1):
        if k > 1:
            y_even = A2.dot(y_even)
        # Compute next even power
        y_even2 = A2.dot(y_even)
        # Index of largest magnitude in y_even
        j = np.argmax(np.abs(y_even))
        # Estimate λ^2
        lambda2_new = y_even2[j] / y_even[j]
        # Compute λ
        lambda1 = np.sqrt(lambda2_new)
        # Compute y_{2k+1}
        y_odd = A.dot(y_even)
        # Form eigenvectors
        x1 = y_even + lambda1 * y_odd
        x2 = y_even - lambda1 * y_odd
        # Normalize
        x1 = x1 / np.linalg.norm(x1, ord=norm_ord)
        x2 = x2 / np.linalg.norm(x2, ord=norm_ord)

        # Record iteration: [k, y_even1..n, lambda2, x1_1..n, x2_1..n]
        row = [k] + y_even.tolist() + [lambda2_new] + x1.tolist() + x2.tolist()
        history.append(row)

        # Convergence check on λ^2
        if k > 1 and abs(lambda2_new - eigen2) < tol:
            break
        eigen2 = lambda2_new

    # Display results in pandas
    cols = ['Iteration'] + [f'y{i+1}' for i in range(n)] + ['lambda^2'] + [f'x1_{i+1}' for i in range(n)] + [f'x2_{i+1}' for i in range(n)]
    df = pd.DataFrame(history, columns=cols)
    if (display==True):
      print(df.to_string(
        index=False,
        float_format=lambda v: f"{v:.{precision}f}"
    ))

    # Final eigenvalue
    lambda1_final = np.sqrt(eigen2)
    return lambda1_final, x1, x2


In [19]:
def compute_all_eigenpairs(
    A: np.ndarray,
    x0: np.ndarray,
    tol: float = 1e-6,
    max_iter: int = 200,
    norm_ord: Union[int, float] = 2,
    precision: int = 8
) -> Tuple[List[float], List[np.ndarray]]:
    n = A.shape[0]
    M = A.copy().astype(float)
    y = x0.astype(float).flatten()
    eigs = []
    vecs = []

    for k in range(n):
        lam, v = power_method(
            M, y, tol=tol, max_iter=max_iter,
            norm_ord=norm_ord, precision=precision, display=False
        )
        eigs.append(lam)
        vecs.append(v)
        M = deflate_once(M, lam, v, y, tol, max_iter, norm_ord, precision)
        y = x0.astype(float).flatten()

    pairs = sorted(zip(eigs, vecs), key=lambda ev: -abs(ev[0]))
    eigs_sorted, vecs_sorted = zip(*pairs)
    return list(eigs_sorted), list(vecs_sorted)


# Khai triển kỳ dị SVD

## Thuật toán

In [20]:
def complete_orthonormal_basis(B: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    """
    Given B (n×k) with orthonormal columns, extend to a full n×n orthonormal basis.
    """
    n, k = B.shape
    Q = np.zeros((n, n))
    Q[:, :k] = B
    for j in range(k, n):
        # start with a random vector
        v = np.random.rand(n)
        # orthonormalize against existing columns
        for i in range(j):
            v -= np.dot(Q[:, i], v) * Q[:, i]
        norm = np.linalg.norm(v)
        # if vector is nearly zero, retry
        while norm < tol:
            v = np.random.rand(n)
            for i in range(j):
                v -= np.dot(Q[:, i], v) * Q[:, i]
            norm = np.linalg.norm(v)
        Q[:, j] = v / norm
    return Q

# Khai triển kỳ dị SVD (Bằng Lũy thừa và Xuống thang)

**Input**: Ma trận $A_{m \times n}$, vector khởi tạo $x^{(0)}$, sai số $\varepsilon$.

**Output**: Các ma trận $U, \Sigma, V^T$ thỏa mãn $A = U\Sigma V^T$.

1. Kiểm tra nếu $m \le n$:

    1.1. Tính ma trận $M = A \cdot A^T$.

    1.2. Áp dụng Thuật toán Lũy thừa kết hợp Xuống thang cho ma trận $M$ để tìm các giá trị riêng $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_m$ và các vector riêng trực chuẩn $u_1, u_2, \dots, u_m$ tương ứng.
    
    1.3. Gán $U = \begin{bmatrix} u_1 & u_2 & \dots & u_m \end{bmatrix}$

    1.4. Tính các giá trị kỳ dị $\sigma_i = \sqrt{\lambda_i}$ với $i = \overline{1,m}$.

    1.5. Khởi tạo ma trận $\Sigma_{m \times n} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_m)$
    
    1.6. Khởi tạo ma trận $V_{n \times n}$. Với mỗi $i$ từ $1 \to m$:

    - Nếu $\sigma_i > 0 \Rightarrow$ Gán cột thứ $i$ của $V$ là $v_i = \dfrac{A^T u_i}{\sigma_i}$.

    - Nếu $\sigma_i = 0 \Rightarrow$ Gán cột thứ $i$ của $V$ là vector $0$.

    1.7. Đặt $r$ là số lượng $\sigma_i > 0$. Thực hiện lần lượt các phép tính và gán:

    - Tính các vector $v_{r+1}, \dots, v_n$ thuộc $\text{ker}(A)$ trực chuẩn với $v_1, \dots, v_r$ bằng Gram-Schmidt.

    - Gán $V = \begin{bmatrix} v_1 & v_2 & \dots & v_n \end{bmatrix}$.

2. Nếu $m > n$:

    2.1. Tính ma trận $N = A^T \cdot A$.

    2.2. Áp dụng Thuật toán Lũy thừa kết hợp Xuống thang cho ma trận $N$ để tìm các giá trị riêng $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_n$ và các vector riêng trực chuẩn $v_1, v_2, \dots, v_n$ tương ứng.
    
    2.3. Gán $V = \begin{bmatrix} v_1 & v_2 & \dots & v_n \end{bmatrix}$

    2.4. Tính các giá trị kỳ dị $\sigma_j = \sqrt{\lambda_j}$ với $j = 1 \dots n$.

    2.5. Khởi tạo ma trận $\Sigma_{m \times n} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_n)$
    
    2.6. Khởi tạo ma trận $U_{m \times m}$. Với mỗi $j$ từ $1 \to n$:

    - Nếu $\sigma_j > 0 \Rightarrow$ Gán cột thứ $j$ của $U$ là $u_j = \dfrac{A v_j}{\sigma_j}$.

    - Nếu $\sigma_j = 0 \Rightarrow$ Gán cột thứ $j$ của $U$ là vector $0$.

    2.7. Đặt $r$ là số lượng $\sigma_j > 0$. Thực hiện lần lượt các phép tính và gán:

    - Tính các vector $u_{r+1}, \dots, u_m$ thuộc $\text{ker}(A^T)$ trực chuẩn với $u_1, \dots, u_r$ bằng Gram-Schmidt.

    - Gán $U = \begin{bmatrix} u_1 & u_2 & \dots & u_m \end{bmatrix}$
    
3. Trả về $U$, $\Sigma$, $V^T$.

In [21]:
def compute_svd_power(A_df: pd.DataFrame, x0=None, tol: float = 1e-10, max_iter=1000):
    A = A_df.values.astype(float)
    m, n = A.shape
    
    if m <= n:
        T = A.dot(A.T)
        print(f"Trường hợp m <= n, sử dụng T = A · Aᵀ ({m}x{m})")
        print("Ma trận M:")
        output_matrix(T)
        
        if x0 is None: x0 = np.ones(m)
        eigs, vecs = compute_all_eigenpairs(T, x0, tol=tol, max_iter=max_iter, precision=4)
        
        for i in range(m):
            print(f"Giá trị riêng λ{i+1} = {eigs[i]}")
            print(f"Vector riêng u{i+1}:")
            output_matrix(vecs[i])
            
        r = sum(1 for e in eigs if e > tol)
        
        U = complete_orthonormal_basis(np.array(vecs).T, tol=tol)
        
        S = np.zeros((m, n))
        for i in range(r):
            S[i, i] = np.sqrt(eigs[i])
            
        print("Các giá trị kỳ dị:")
        for i in range(r):
            print(f"σ{i+1} = {S[i,i]}")
        for i in range(r, m):
            print(f"σ{i+1} = 0.0")
            
        print("Ma trận Σ:")
        output_matrix(S)
            
        Vt = np.zeros((n, n))
        for i in range(r):
            Vt[i, :] = A.T.dot(U[:, i]) / S[i, i]
            
        Vt_full = complete_orthonormal_basis(Vt[:r, :].T, tol=tol).T
        
        for i in range(r):
            print(f"Vector kỳ dị phải v{i+1}:")
            output_matrix(Vt_full[i, :])
        
        return pd.DataFrame(U), pd.DataFrame(S), pd.DataFrame(Vt_full)
        
    else:
        T = A.T.dot(A)
        print(f"Trường hợp m > n, sử dụng T = Aᵀ · A ({n}x{n})")
        print("Ma trận M:")
        output_matrix(T)
        
        if x0 is None: x0 = np.ones(n)
        eigs, vecs = compute_all_eigenpairs(T, x0, tol=tol, max_iter=max_iter, precision=4)
        
        for i in range(n):
            print(f"Giá trị riêng λ{i+1} = {eigs[i]}")
            print(f"Vector riêng v{i+1}:")
            output_matrix(vecs[i])
            
        r = sum(1 for e in eigs if e > tol)
        
        Vt = complete_orthonormal_basis(np.array(vecs).T, tol=tol).T
        
        S = np.zeros((m, n))
        for i in range(r):
            S[i, i] = np.sqrt(eigs[i])
            
        print("Các giá trị kỳ dị:")
        for i in range(r):
            print(f"σ{i+1} = {S[i,i]}")
        for i in range(r, n):
            print(f"σ{i+1} = 0.0")
            
        print("Ma trận Σ:")
        output_matrix(S)
            
        U = np.zeros((m, m))
        for i in range(r):
            U[:, i] = A.dot(Vt[i, :].T) / S[i, i]
            
        U_full = complete_orthonormal_basis(U[:, :r], tol=tol)
        
        for i in range(r):
            print(f"Vector kỳ dị trái u{i+1}:")
            output_matrix(U_full[:, i])
        
        return pd.DataFrame(U_full), pd.DataFrame(S), pd.DataFrame(Vt)


# Thuật toán SVD rút gọn (Bằng Lũy thừa và Xuống thang)

**Input**: Ma trận $A_{m \times n}$, vector khởi tạo $x^{(0)}$, sai số $\varepsilon$.

**Output**: Khai triển kỳ dị rút gọn $A = U_r \Sigma_r V_r^T$.

1. Kiểm tra nếu $m \le n$:

    1.1. Tính ma trận $M = A \cdot A^T$.

    1.2. Áp dụng Thuật toán Lũy thừa kết hợp Xuống thang cho ma trận $M$ để tìm các giá trị riêng dương $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_r > 0$, với $r$ là hạng của $A$. Lấy các vector riêng trực chuẩn $u_1, u_2, \dots, u_r$ tương ứng.
    
    1.3. Gán $U_r = \begin{bmatrix} u_1 & u_2 & \dots & u_r \end{bmatrix}$.

    1.4. Tính các giá trị kỳ dị $\sigma_i = \sqrt{\lambda_i}$ với $i = \overline{1,r}$.

    1.5. Khởi tạo ma trận đường chéo $\Sigma_{r} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_r)$.
    
    1.6. Khởi tạo ma trận $V_r^T$ kích thước $r \times n$. Với mỗi $i$ từ $1 \to r$:

    - Gán hàng thứ $i$ của $V_r^T$ là $v_i^T = \dfrac{u_i^T A}{\sigma_i}$.

2. Nếu $m > n$:

    2.1. Tính ma trận $N = A^T \cdot A$.

    2.2. Áp dụng Thuật toán Lũy thừa kết hợp Xuống thang cho ma trận $N$ để tìm các giá trị riêng dương $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_r > 0$, với $r$ là hạng của $A$. Lấy các vector riêng trực chuẩn $v_1, v_2, \dots, v_r$ tương ứng.
    
    2.3. Gán $V_r^T = \begin{bmatrix} v_1 & v_2 & \dots & v_r \end{bmatrix}^T$.

    2.4. Tính các giá trị kỳ dị $\sigma_j = \sqrt{\lambda_j}$ với $j = \overline{1,r}$.

    2.5. Khởi tạo ma trận đường chéo $\Sigma_{r} = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_r)$.
    
    2.6. Khởi tạo ma trận $U_r$ kích thước $m \times r$. Với mỗi $j$ từ $1 \to r$:

    - Gán cột thứ $j$ của $U_r$ là $u_j = \dfrac{A v_j}{\sigma_j}$.

3. Trả về $U_r$, $\Sigma_r$, $V_r^T$.

In [22]:
def compute_reduced_svd_power(A_df: pd.DataFrame, x0=None, tol: float = 1e-10, max_iter=1000):
    A = A_df.values.astype(float)
    m, n = A.shape
    
    if m <= n:
        T = A.dot(A.T)
        print(f"Trường hợp m <= n, sử dụng T = A · Aᵀ ({m}x{m})")
        if x0 is None: x0 = np.ones(m)
        eigs, vecs = compute_all_eigenpairs(T, x0, tol=tol, max_iter=max_iter, precision=4)
        
        r = sum(1 for e in eigs if e > tol)
        print(f"Hạng r = {r}")
        
        sigmas = np.sqrt([e for e in eigs[:r]])
        U_r = np.array(vecs[:r]).T
        Sigma_r = np.diag(sigmas)
        
        print("Ma trận U_r:")
        output_matrix(U_r)
        print("Ma trận Σ_r:")
        output_matrix(Sigma_r)
        
        Vt_r = np.zeros((r, n))
        for i in range(r):
            Vt_r[i, :] = A.T.dot(U_r[:, i]) / sigmas[i]
            
        print("Ma trận V_r^T:")
        output_matrix(Vt_r)
            
    else:
        T = A.T.dot(A)
        print(f"Trường hợp m > n, sử dụng T = Aᵀ · A ({n}x{n})")
        if x0 is None: x0 = np.ones(n)
        eigs, vecs = compute_all_eigenpairs(T, x0, tol=tol, max_iter=max_iter, precision=4)
        
        r = sum(1 for e in eigs if e > tol)
        print(f"Hạng của ma trận A: r = {r}")
        
        sigmas = np.sqrt([e for e in eigs[:r]])
        Vt_r = np.array(vecs[:r])
        Sigma_r = np.diag(sigmas)
        
        U_r = np.zeros((m, r))
        for i in range(r):
            U_r[:, i] = A.dot(Vt_r[i, :].T) / sigmas[i]
            
        print("Ma trận U_r:")
        output_matrix(U_r)
        print("Ma trận Σ_r:")
        output_matrix(Sigma_r)
        print("Ma trận V_r^T:")
        output_matrix(Vt_r)
            
    return pd.DataFrame(U_r), pd.DataFrame(Sigma_r), pd.DataFrame(Vt_r)


# 3. Tính số điều kiện và Xấp xỉ ma trận

**A. Thuật toán Tính số điều kiện**

**Input**: Tập $r$ giá trị kỳ dị $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$ thu được từ Lũy thừa và Xuống thang.

**Output**: Số điều kiện $\kappa(A)$.

1. Tính $\kappa(A) = \dfrac{\sigma_1}{\sigma_r}$. In ra $\kappa(A)$.

**B. Thuật toán Xấp xỉ ma trận hạng k**

**Input**: Các ma trận $U_r, \Sigma_r, V_r^T$ thu được từ SVD rút gọn, và số nguyên $k \le r$.

**Output**: Ma trận xấp xỉ $A_k$.

1. Đặt $A_k = [0]_{m \times n}$.

2. Lặp $i$ từ $1$ đến $k$:

    - Tính $A_k = A_k + \sigma_i u_i v_i^T$.
    
3. In ra $A_k$ và sai số xấp xỉ chuẩn 2 là $\sigma_{k+1}$ (hoặc 0 nếu $k=r$).


In [23]:
def analyze_svd_power(A, U_r, Sigma_r, Vt_r, k=None, tol=1e-10, precision=4):
    if hasattr(A, 'values'): A = A.values.astype(float)
    if hasattr(U_r, 'values'): U_r = U_r.values.astype(float)
    if hasattr(Sigma_r, 'values'): Sigma_r = Sigma_r.values.astype(float)
    if hasattr(Vt_r, 'values'): Vt_r = Vt_r.values.astype(float)
    
    m, n = A.shape
    sigmas = np.diag(Sigma_r)
    r = len(sigmas)
    
    print(f"\n--- 1. TÍNH SỐ ĐIỀU KIỆN ---")
    if r > 0:
        cond_num = sigmas[0] / sigmas[-1]
        print(f"Số điều kiện κ(A) = σ_1 / σ_r = {sigmas[0]:.{precision}f} / {sigmas[-1]:.{precision}f} = {cond_num:.{precision}f}")
    else:
        print("Ma trận 0.")
        
    print(f"\n--- 2. XẤP XỈ MA TRẬN (LOW-RANK APPROXIMATION) ---")
    if k is not None:
        if k > r: k = r
        print(f"Xấp xỉ ma trận A với k = {k}:")
        Ak = np.zeros_like(A)
        for i in range(k):
            u_i = U_r[:, i].reshape(-1, 1)
            vt_i = Vt_r[i, :].reshape(1, -1)
            Ak += sigmas[i] * u_i.dot(vt_i)
        print(f"Ma trận A_{k}:")
        output_matrix(Ak, precision)
        
        if k < r:
            print(f"Sai số xấp xỉ chuẩn 2 = σ_{k+1} = {sigmas[k]:.{precision}f}")
        else:
            print("Sai số xấp xỉ chuẩn 2 = 0")



In [24]:
A_df = read_matrix("SVD_input_A.txt")

print("\n===========================")
print("1. TÍNH SVD ĐẦY ĐỦ BẰNG LŨY THỪA")
print("===========================")
U, S, Vt = compute_svd_power(A_df, tol=1e-9)

print("--- Ma trận A ---")
output_matrix(A_df)
print("--- Ma trận U ---")
output_matrix(U)
print("--- Ma trận Σ ---")
output_matrix(S)
print("--- Ma trận V^T ---")
output_matrix(Vt)

print("\n===========================")
print("2. TÍNH SVD RÚT GỌN BẰNG LŨY THỪA")
print("===========================")
Ur, Sr, Vtr = compute_reduced_svd_power(A_df, tol=1e-9)


print("\n===========================")
print("3. TÍNH SỐ ĐIỀU KIỆN & XẤP XỈ")
print("===========================")
analyze_svd_power(A_df, Ur, Sr, Vtr, k=2, precision=4)




1. TÍNH SVD ĐẦY ĐỦ BẰNG LŨY THỪA
Trường hợp m > n, sử dụng T = Aᵀ · A (3x3)
Ma trận M:


<IPython.core.display.Math object>

Giá trị riêng λ1 = 345.18097878651497
Vector riêng v1:


<IPython.core.display.Math object>

Giá trị riêng λ2 = 43.40005304435462
Vector riêng v2:


...

Ma trận V_r^T:


<IPython.core.display.Math object>


3. TÍNH SỐ ĐIỀU KIỆN & XẤP XỈ

--- 1. TÍNH SỐ ĐIỀU KIỆN ---
Số điều kiện κ(A) = σ_1 / σ_r = 18.5790 / 2.1021 = 8.8382

--- 2. XẤP XỈ MA TRẬN (LOW-RANK APPROXIMATION) ---
Xấp xỉ ma trận A với k = 2:
Ma trận A_2:


<IPython.core.display.Math object>

Sai số xấp xỉ chuẩn 2 = σ_3 = 2.1021
